# Training Source Notebook

Original Colab notebook used to train the Layer-2 model weights shipped with this repository:

- `multiscale_cnn_bottom.pt`, `multiscale_cnn_top.pt` (Layer-2 CNN)
- `lgb_bottom_v1`, `lgb_top_v1` (Layer-2 meta-label LightGBM)

Layer-1 volatility LightGBM weights (`lightgbm_v3_flat`) were trained by a separate script not included in this release.


In [ ]:
import os
from pathlib import Path

# Original training environment was Google Colab + Google Drive.
# On Colab, mount Drive and chdir into the thesis-data root.
# Locally, the user should cd into the directory containing `processed/`,
# `splits/`, `features/` before launching the notebook.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    os.chdir("/content/drive/MyDrive/毕设/data")
except ImportError:
    # Not on Colab — assume the user has already set CWD appropriately
    # (the `processed/` and `splits/` subdirs must be visible from here).
    pass
print("CWD:", Path.cwd())


# mins

In [ ]:
#!/usr/bin/env python3
"""
resample_1min_to_15min.py
─────────────────────────
Reads 1-minute OHLCV CSVs from processed/, resamples to 15-minute bars,
computes 53 technical indicators (matching the 1-hour feature set), and
saves both the resampled OHLCV and the full feature files.

Usage:
    python resample_1min_to_15min.py
"""

import os
import sys
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# ──────────────────────────────────────────────────────────────────────
# CONFIG
# ──────────────────────────────────────────────────────────────────────
CONFIG = {
    "PROCESSED_DIR": "processed",
    "FEATURES_DIR": "features",
    "TICKERS": ["AAPL", "MSFT", "GOOGL", "GOOG", "NVDA", "TSLA", "SPY", "QQQ"],
    "RESAMPLE_FREQ": "15min",
}

# Possible names for the timestamp column in source CSVs
TIMESTAMP_ALIASES = ["timestamp", "ts_event", "datetime", "date", "time", "ts"]


# ──────────────────────────────────────────────────────────────────────
# 1. Discover reference indicator columns from an existing 1hour file
# ──────────────────────────────────────────────────────────────────────
def get_reference_indicator_cols(features_dir: str) -> list[str]:
    ohlcv_cols = {"timestamp", "open", "high", "low", "close", "volume"}
    for fname in sorted(os.listdir(features_dir)):
        if fname.endswith("_1hour_features.csv"):
            path = os.path.join(features_dir, fname)
            df = pd.read_csv(path, nrows=0)
            ind_cols = [c for c in df.columns if c.lower() not in ohlcv_cols]
            if ind_cols:
                print(f"[ref] Using {fname} as reference → {len(ind_cols)} indicator columns")
                return ind_cols
    return []


# ──────────────────────────────────────────────────────────────────────
# 2. Detect and normalize the timestamp column
# ──────────────────────────────────────────────────────────────────────
def find_time_column(df: pd.DataFrame) -> str | None:
    """Find the timestamp column regardless of naming convention."""
    cols_lower = {c.lower(): c for c in df.columns}
    for alias in TIMESTAMP_ALIASES:
        if alias in cols_lower:
            return cols_lower[alias]
    return None


def normalize_timestamps(df: pd.DataFrame) -> pd.DataFrame:
    """
    Find the time column, parse as datetime, set as index named 'timestamp'.
    Strips timezone to naive UTC for clean resampling.
    """
    time_col = find_time_column(df)

    if time_col is not None:
        df = df.copy()
        df[time_col] = pd.to_datetime(df[time_col], utc=True)
        df[time_col] = df[time_col].dt.tz_localize(None)
        df = df.set_index(time_col)
        df.index.name = "timestamp"
    else:
        df = df.copy()
        df.index = pd.to_datetime(df.index, utc=True).tz_localize(None)
        df.index.name = "timestamp"

    if not isinstance(df.index, pd.DatetimeIndex):
        raise ValueError(f"Could not create DatetimeIndex. Columns: {list(df.columns)}")

    return df


# ──────────────────────────────────────────────────────────────────────
# 3. Resample 1-min → 15-min OHLCV
# ──────────────────────────────────────────────────────────────────────
def resample_ohlcv(df_1min: pd.DataFrame, freq: str) -> pd.DataFrame:
    """Standard OHLCV aggregation on a DatetimeIndex dataframe."""
    df_1min = normalize_timestamps(df_1min)

    # Lowercase column names
    df_1min.columns = [c.lower() for c in df_1min.columns]

    agg_rules = {
        "open": "first",
        "high": "max",
        "low": "min",
        "close": "last",
        "volume": "sum",
    }
    agg_rules = {k: v for k, v in agg_rules.items() if k in df_1min.columns}

    df_15 = df_1min[list(agg_rules.keys())].resample(freq).agg(agg_rules)

    # Drop non-trading periods (all NaN) and rows without close
    df_15.dropna(how="all", inplace=True)
    df_15.dropna(subset=["close"], inplace=True)

    return df_15


# ──────────────────────────────────────────────────────────────────────
# 4. Compute technical indicators via pandas_ta
# ──────────────────────────────────────────────────────────────────────
def compute_indicators_pandas_ta(df: pd.DataFrame, ref_cols: list[str]) -> pd.DataFrame:
    try:
        import pandas_ta as ta
    except ImportError:
        print("[WARN] pandas_ta not installed – pip install pandas_ta")
        return compute_indicators_manual(df, ref_cols)

    work = df[["open", "high", "low", "close", "volume"]].copy()

    # Reverting to original call for pandas_ta accessor.
    # The previous error indicates that `ta.strategy()` directly on the module is incorrect.
    # The root cause might be an issue with the pandas_ta accessor registration itself due to dependency conflicts.
    work.ta.strategy("All")

    ta_cols_lower = {c.lower(): c for c in work.columns}
    matched = {}
    unmatched = []

    for ref in ref_cols:
        rl = ref.lower()
        if rl in ta_cols_lower:
            matched[ref] = ta_cols_lower[rl]
        else:
            found = False
            for tc_lower, tc_orig in ta_cols_lower.items():
                if rl.replace("_", "") == tc_lower.replace("_", ""):
                    matched[ref] = tc_orig
                    found = True
                    break
            if not found:
                unmatched.append(ref)

    result = df[["open", "high", "low", "close", "volume"]].copy()
    for ref_name, ta_name in matched.items():
        result[ref_name] = work[ta_name]

    if unmatched:
        manual = _compute_manual_indicators(df)
        manual_lower = {c.lower(): c for c in manual.columns}
        still_missing = []
        for ref in unmatched:
            rl = ref.lower()
            if rl in manual_lower:
                result[ref] = manual[manual_lower[rl]]
            else:
                still_missing.append(ref)
        if still_missing:
            print(f"  [WARN] {len(still_missing)} indicators could not be matched: "
                  f"{still_missing[:10]}{'...' if len(still_missing) > 10 else ''}")

    return result


def _compute_manual_indicators(df: pd.DataFrame) -> pd.DataFrame:
    out = pd.DataFrame(index=df.index)
    c, h, l, v = df["close"], df["high"], df["low"], df["volume"]
    o = df["open"]

    # Moving Averages
    for w in [5, 10, 20, 50, 100, 200]:
        out[f"sma_{w}"] = c.rolling(w).mean()
        out[f"ema_{w}"] = c.ewm(span=w, adjust=False).mean()

    # MACD
    ema12 = c.ewm(span=12, adjust=False).mean()
    ema26 = c.ewm(span=26, adjust=False).mean()
    out["macd"] = ema12 - ema26
    out["macd_signal"] = out["macd"].ewm(span=9, adjust=False).mean()
    out["macd_hist"] = out["macd"] - out["macd_signal"]

    # RSI
    for w in [7, 14, 21]:
        delta = c.diff()
        gain = delta.clip(lower=0).rolling(w).mean()
        loss = (-delta.clip(upper=0)).rolling(w).mean()
        rs = gain / loss.replace(0, np.nan)
        out[f"rsi_{w}"] = 100 - (100 / (1 + rs))

    # Bollinger Bands
    for w in [20]:
        mid = c.rolling(w).mean()
        std = c.rolling(w).std()
        out[f"bb_upper_{w}"] = mid + 2 * std
        out[f"bb_middle_{w}"] = mid
        out[f"bb_lower_{w}"] = mid - 2 * std
        out[f"bb_bandwidth_{w}"] = (out[f"bb_upper_{w}"] - out[f"bb_lower_{w}"]) / mid
        out[f"bb_percent_{w}"] = (c - out[f"bb_lower_{w}"]) / (out[f"bb_upper_{w}"] - out[f"bb_lower_{w}"]).replace(0, np.nan)

    out["bb_upper"] = out.get("bb_upper_20", c)
    out["bb_middle"] = out.get("bb_middle_20", c)
    out["bb_lower"] = out.get("bb_lower_20", c)

    # ATR
    for w in [7, 14, 21]:
        tr = pd.concat([
            h - l, (h - c.shift(1)).abs(), (l - c.shift(1)).abs()
        ], axis=1).max(axis=1)
        out[f"atr_{w}"] = tr.rolling(w).mean()
    out["true_range"] = pd.concat([
        h - l, (h - c.shift(1)).abs(), (l - c.shift(1)).abs()
    ], axis=1).max(axis=1)

    # Stochastic
    for w in [14]:
        low_min = l.rolling(w).min()
        high_max = h.rolling(w).max()
        out[f"stoch_k_{w}"] = 100 * (c - low_min) / (high_max - low_min).replace(0, np.nan)
        out[f"stoch_d_{w}"] = out[f"stoch_k_{w}"].rolling(3).mean()

    # ADX
    for w in [14]:
        plus_dm = h.diff().clip(lower=0)
        minus_dm = (-l.diff()).clip(upper=0)
        tr = pd.concat([h - l, (h - c.shift(1)).abs(), (l - c.shift(1)).abs()], axis=1).max(axis=1)
        atr = tr.rolling(w).mean()
        plus_di = 100 * (plus_dm.rolling(w).mean() / atr.replace(0, np.nan))
        minus_di = 100 * (minus_dm.rolling(w).mean() / atr.replace(0, np.nan))
        dx = 100 * ((plus_di - minus_di).abs() / (plus_di + minus_di).replace(0, np.nan))
        out[f"adx_{w}"] = dx.rolling(w).mean()
        out[f"plus_di_{w}"] = plus_di
        out[f"minus_di_{w}"] = minus_di

    # CCI
    for w in [14, 20]:
        tp = (h + l + c) / 3
        out[f"cci_{w}"] = (tp - tp.rolling(w).mean()) / (0.015 * tp.rolling(w).std())

    # Williams %R
    for w in [14]:
        high_max = h.rolling(w).max()
        low_min = l.rolling(w).min()
        out[f"willr_{w}"] = -100 * (high_max - c) / (high_max - low_min).replace(0, np.nan)

    # MFI
    for w in [14]:
        tp = (h + l + c) / 3
        mf = tp * v
        pos_mf = pd.Series(np.where(tp > tp.shift(1), mf, 0), index=df.index).rolling(w).sum()
        neg_mf = pd.Series(np.where(tp < tp.shift(1), mf, 0), index=df.index).rolling(w).sum()
        mfi = 100 - (100 / (1 + pos_mf / neg_mf.replace(0, np.nan)))
        out[f"mfi_{w}"] = mfi

    # OBV
    obv = pd.Series(np.where(c > c.shift(1), v, np.where(c < c.shift(1), -v, 0)),
                    index=df.index).cumsum()
    out["obv"] = obv

    # VWAP
    tp = (h + l + c) / 3
    out["vwap"] = (tp * v).cumsum() / v.cumsum().replace(0, np.nan)

    # Volatility / momentum
    for w in [5, 10, 20]:
        out[f"volatility_{w}"] = c.pct_change().rolling(w).std()
        out[f"return_{w}"] = c.pct_change(w)
        out[f"log_return_{w}"] = np.log(c / c.shift(w))

    out["return_1"] = c.pct_change(1)
    out["log_return_1"] = np.log(c / c.shift(1))

    # Price ratios / spreads
    out["hl_spread"] = (h - l) / c.replace(0, np.nan)
    out["oc_spread"] = (c - o) / o.replace(0, np.nan)

    # Volume indicators
    for w in [5, 10, 20]:
        out[f"volume_sma_{w}"] = v.rolling(w).mean()
    out["volume_ratio"] = v / v.rolling(20).mean().replace(0, np.nan)

    # Time features
    if isinstance(df.index, pd.DatetimeIndex):
        out["hour"] = df.index.hour
        out["day_of_week"] = df.index.dayofweek
        out["minute"] = df.index.minute
    else:
        out["hour"] = 0
        out["day_of_week"] = 0
        out["minute"] = 0

    return out


def compute_indicators_manual(df: pd.DataFrame, ref_cols: list[str]) -> pd.DataFrame:
    manual = _compute_manual_indicators(df)
    result = df[["open", "high", "low", "close", "volume"]].copy()

    manual_lower = {c.lower(): c for c in manual.columns}
    matched = 0
    for ref in ref_cols:
        rl = ref.lower()
        if rl in manual_lower:
            result[ref] = manual[manual_lower[rl]]
            matched += 1
        else:
            for ml, mc in manual_lower.items():
                if rl.replace("_", "") == ml.replace("_", ""):
                    result[ref] = manual[mc]
                    matched += 1
                    break

    if matched < len(ref_cols) * 0.5:
        print(f"  [INFO] Low match rate ({matched}/{len(ref_cols)}), adding all manual indicators")
        for col in manual.columns:
            if col not in result.columns:
                result[col] = manual[col]

    return result


# ──────────────────────────────────────────────────────────────────────
# 5. Main pipeline
# ──────────────────────────────────────────────────────────────────────
def process_ticker(ticker: str, config: dict, ref_cols: list[str]) -> dict:
    proc_dir = config["PROCESSED_DIR"]
    feat_dir = config["FEATURES_DIR"]
    freq = config["RESAMPLE_FREQ"]

    in_path = os.path.join(proc_dir, f"{ticker}_1min.csv")
    if not os.path.exists(in_path):
        print(f"[SKIP] {in_path} not found")
        return {}

    # ── Read ────────────────────────────────────────────────────────
    df_1min = pd.read_csv(in_path)
    n_1min = len(df_1min)

    # ── Resample ────────────────────────────────────────────────────
    df_15 = resample_ohlcv(df_1min, freq)
    n_15 = len(df_15)

    if n_15 == 0:
        print(f"[ERROR] {ticker}: resample produced 0 rows!")
        return {}

    # Sanity check
    expected = n_1min // 15
    if n_15 < expected * 0.5:
        print(f"  [WARN] {ticker}: got {n_15} 15min bars from {n_1min} 1min bars "
              f"(expected ~{expected})")

    # Save resampled OHLCV
    out_ohlcv = os.path.join(proc_dir, f"{ticker}_15min.csv")
    df_15.to_csv(out_ohlcv)

    # ── Compute indicators ──────────────────────────────────────────
    if ref_cols:
        df_feat = compute_indicators_pandas_ta(df_15, ref_cols)
    else:
        manual = _compute_manual_indicators(df_15)
        df_feat = pd.concat([df_15[["open", "high", "low", "close", "volume"]], manual], axis=1)

    # Drop duplicate columns
    df_feat = df_feat.loc[:, ~df_feat.columns.duplicated()]

    n_features = len([c for c in df_feat.columns
                      if c.lower() not in {"open", "high", "low", "close", "volume", "timestamp"}])

    # Save features
    out_feat = os.path.join(feat_dir, f"{ticker}_15min_features.csv")
    df_feat.index.name = "timestamp"
    df_feat.to_csv(out_feat)

    print(f"  {ticker}: {n_1min:>9,} 1min rows → {n_15:>7,} 15min rows → {n_features} indicator cols")
    return {"ticker": ticker, "n_1min": n_1min, "n_15min": n_15, "n_features": n_features}


def main():
    config = CONFIG
    proc_dir = config["PROCESSED_DIR"]
    feat_dir = config["FEATURES_DIR"]

    if not os.path.isdir(proc_dir):
        print(f"[ERROR] Processed dir not found: {proc_dir}")
        sys.exit(1)
    os.makedirs(feat_dir, exist_ok=True)

    ref_cols = get_reference_indicator_cols(feat_dir)
    if not ref_cols:
        print("[INFO] No 1hour_features reference found – will compute all indicators manually")

    print(f"\nResampling 1min → {config['RESAMPLE_FREQ']} + computing indicators")
    print("=" * 75)
    results = []
    for ticker in config["TICKERS"]:
        info = process_ticker(ticker, config, ref_cols)
        if info:
            results.append(info)

    print("=" * 75)
    if results:
        print(f"Done: {len(results)} tickers processed")
        total_1 = sum(r["n_1min"] for r in results)
        total_15 = sum(r["n_15min"] for r in results)
        print(f"Total: {total_1:,} 1min rows → {total_15:,} 15min rows")
    else:
        print("No tickers processed. Check that processed/{TICKER}_1min.csv files exist.")


if __name__ == "__main__":
    main()

# mo


In [ ]:
"""
multiscale_cnn.py
=================
Multi-Scale CNN for turning point detection.

Short branch: 30×15min bars (7.5h) — immediate reversal patterns
Long branch:  48×1hour bars (2 days) — bull/bear dynamics, momentum decay,
              volume divergence, volatility contraction

Three evaluation levels:
  1. Detection metrics (precision/recall/AUC) vs single-scale CNN
  2. MFE/MAE analysis — do multi-scale candidates have better trade quality?
  3. Meta-label integration — LightGBM filter on multi-scale candidates

Usage (Colab A100):
    !python multiscale_cnn.py
"""

import os, warnings, time, json, math
import numpy as np
import pandas as pd
from scipy import stats as sp_stats
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

try:
    import lightgbm as lgb
    HAS_LGB = True
except ImportError:
    HAS_LGB = False

# ─── Reproducibility ───
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ─────────────────────────── CONFIG ───────────────────────────

CFG = {
    "FEATURES_DIR": "features",
    "RESULTS_DIR": "results",
    "TICKERS": ["AAPL", "MSFT", "GOOGL", "GOOG", "NVDA", "TSLA", "SPY", "QQQ"],
    "SHORT_FREQ": "15min",
    "LONG_FREQ": "1hour",
    "SHORT_WIN": 30,
    "LONG_WIN": 48,
    # Zigzag
    "ZIGZAG_ATR_MULT": 2.0,
    "ZIGZAG_ATR_PERIOD": 14,
    "MIN_BARS_BETWEEN": 6,
    "TP_PROXIMITY": 3,
    "NEG_RATIO": 3,
    # Training
    "BATCH": 256,
    "LR": 1e-3,
    "WD": 1e-4,
    "EPOCHS": 100,
    "PATIENCE": 20,
    "SCHED_PATIENCE": 10,
    "GRAD_CLIP": 1.0,
    # Eval
    "DET_THRESHOLDS": [0.3, 0.4, 0.5, 0.6, 0.7, 0.8],
    "MFE_MAX_BARS": [12, 24, 48],
    # Meta-label
    "META_TP": 0.005,
    "META_SL": 0.003,
    "META_MB": 48,
    "LGB_PARAMS": {
        "objective": "binary", "metric": "binary_logloss",
        "n_estimators": 500, "max_depth": 5, "learning_rate": 0.05,
        "subsample": 0.8, "colsample_bytree": 0.8,
        "min_child_samples": 20, "reg_alpha": 0.1, "reg_lambda": 1.0,
        "verbose": -1, "random_state": 42,
    },
    "LGB_THRESHOLDS": [0.45, 0.50, 0.55, 0.60],
    # Single-scale baselines (from previous runs)
    "SINGLE_BASELINES": {
        "bottom_auc": 0.822, "top_auc": 0.798,
        "bottom_prec_07": 0.085, "top_prec_07": 0.081,
        "bottom_mfe_mae_24": 0.69, "top_mfe_mae_24": 0.78,
        "bottom_lgb_wr_05": 0.818, "top_lgb_wr_05": 0.459,
    },
    "OUTPUT_DIR": "results",
    # Set to a list of tasks to skip L1/L2 for (loads saved predictions).
    # E.g. ["bottom"] to skip bottom but run top fully.
    # Set to True to skip all, False to skip none.
    "SKIP_L1_L2": ["bottom", "top"],
}


# ═══════════════════════════════════════════════════════════════
#  DATA LOADING
# ═══════════════════════════════════════════════════════════════

def load_csv(ticker, freq, cfg):
    for f in [freq, "1hour" if freq == "1h" else freq]:
        p = os.path.join(cfg["FEATURES_DIR"], f"{ticker}_{f}_features.csv")
        if os.path.exists(p):
            df = pd.read_csv(p)
            for col in ["timestamp", "ts_event", "datetime", "date"]:
                if col in df.columns:
                    df["timestamp"] = pd.to_datetime(df[col], utc=True)
                    break
            if "timestamp" not in df.columns:
                first = df.columns[0]
                if first not in df.select_dtypes(include=[np.number]).columns:
                    df["timestamp"] = pd.to_datetime(df[first], utc=True)
            df = df.sort_values("timestamp").reset_index(drop=True)
            return df
    raise FileNotFoundError(f"No {freq} features for {ticker}")


def numeric_cols(df, exclude=None):
    ex = {"timestamp", "ts_event", "datetime", "date", "time",
          "unnamed: 0", "symbol", "ticker"}
    if exclude:
        ex |= set(c.lower() for c in exclude)
    return [c for c in df.select_dtypes(include=[np.number]).columns
            if c.lower() not in ex]


# ═══════════════════════════════════════════════════════════════
#  1-HOUR DYNAMICS FEATURES
# ═══════════════════════════════════════════════════════════════

def add_dynamics_features(df):
    """Add 8 'dynamics' columns to 1hour DataFrame. Uses only past data."""
    close = df["close"].values.astype(float)
    open_ = df["open"].values.astype(float) if "open" in df.columns else close
    high = df["high"].values.astype(float) if "high" in df.columns else close
    low = df["low"].values.astype(float) if "low" in df.columns else close
    vol = df["volume"].values.astype(float) if "volume" in df.columns else np.ones(len(df))
    n = len(df)

    # 1. buyer_seller_ratio (rolling 12)
    up_bar = (close > open_).astype(float)
    down_bar = (close < open_).astype(float)
    bsr = np.full(n, 1.0)
    for i in range(12, n):
        seg_up = up_bar[i - 12:i]
        seg_dn = down_bar[i - 12:i]
        seg_vol = vol[i - 12:i]
        up_vol = np.mean(seg_vol[seg_up == 1]) if seg_up.sum() > 0 else 0
        dn_vol = np.mean(seg_vol[seg_dn == 1]) if seg_dn.sum() > 0 else 1e-10
        bsr[i] = up_vol / (dn_vol + 1e-10)
    df["dyn_buyer_seller_ratio"] = bsr

    # 2. rsi_slope_12
    rsi_col = None
    for c in df.columns:
        if c.lower() in ("rsi_14", "rsi"):
            rsi_col = c
            break
    rsi_slope = np.zeros(n)
    if rsi_col is not None:
        rsi = df[rsi_col].values.astype(float)
        x = np.arange(12, dtype=float)
        x_mean = x.mean()
        x_var = ((x - x_mean) ** 2).sum()
        for i in range(12, n):
            seg = rsi[i - 12:i]
            if np.any(np.isnan(seg)):
                continue
            rsi_slope[i] = np.sum((x - x_mean) * (seg - seg.mean())) / (x_var + 1e-10)
    df["dyn_rsi_slope_12"] = rsi_slope

    # 3. macd_accel
    macd_hist_col = None
    for c in df.columns:
        cl = c.lower()
        if "macd" in cl and ("hist" in cl or "diff" in cl):
            macd_hist_col = c
            break
    macd_accel = np.zeros(n)
    if macd_hist_col is not None:
        mh = df[macd_hist_col].values.astype(float)
        for i in range(3, n):
            if not np.isnan(mh[i]) and not np.isnan(mh[i - 3]):
                macd_accel[i] = mh[i] - mh[i - 3]
    df["dyn_macd_accel"] = macd_accel

    # 4. volume_trend (slope of log(vol) over 12 bars)
    vol_trend = np.zeros(n)
    log_vol = np.log(vol + 1)
    x12 = np.arange(12, dtype=float)
    x12m = x12.mean()
    x12v = ((x12 - x12m) ** 2).sum()
    for i in range(12, n):
        seg = log_vol[i - 12:i]
        vol_trend[i] = np.sum((x12 - x12m) * (seg - seg.mean())) / (x12v + 1e-10)
    df["dyn_volume_trend"] = vol_trend

    # 5. atr_change_12
    atr_col = None
    for c in df.columns:
        if c.lower() in ("atr_14", "atr"):
            atr_col = c
            break
    atr_change = np.zeros(n)
    if atr_col is not None:
        atr = df[atr_col].values.astype(float)
        for i in range(12, n):
            if atr[i - 12] > 0 and not np.isnan(atr[i]) and not np.isnan(atr[i - 12]):
                atr_change[i] = (atr[i] - atr[i - 12]) / (atr[i - 12] + 1e-10)
    df["dyn_atr_change_12"] = atr_change

    # 6. lower_shadow_ratio (rolling 6)
    body_low = np.minimum(open_, close)
    full_range = high - low + 1e-10
    lower_shadow = (body_low - low) / full_range
    lsr = np.zeros(n)
    for i in range(6, n):
        lsr[i] = np.mean(lower_shadow[i - 6:i])
    df["dyn_lower_shadow_ratio"] = lsr

    # 7. price_position_48
    pp48 = np.full(n, 0.5)
    for i in range(48, n):
        hh = np.max(high[i - 48:i + 1])
        ll = np.min(low[i - 48:i + 1])
        rng = hh - ll
        pp48[i] = (close[i] - ll) / (rng + 1e-10) if rng > 0 else 0.5
    df["dyn_price_position_48"] = pp48

    # 8. consecutive_down
    consec = np.zeros(n)
    for i in range(1, n):
        if close[i] < close[i - 1]:
            consec[i] = consec[i - 1] + 1
        else:
            consec[i] = 0
    df["dyn_consecutive_down"] = consec

    return df


# ═══════════════════════════════════════════════════════════════
#  ZIGZAG LABELLING (ATR-adaptive, on 15min)
# ═══════════════════════════════════════════════════════════════

def compute_atr(high, low, close, period=14):
    n = len(close)
    tr = np.zeros(n)
    tr[0] = high[0] - low[0]
    for i in range(1, n):
        tr[i] = max(high[i] - low[i],
                     abs(high[i] - close[i - 1]),
                     abs(low[i] - close[i - 1]))
    atr = np.full(n, np.nan)
    if n >= period:
        atr[period - 1] = np.mean(tr[:period])
        for i in range(period, n):
            atr[i] = (atr[i - 1] * (period - 1) + tr[i]) / period
    return atr


def zigzag_atr(high, low, close, atr_mult=2.0, atr_period=14,
               min_bars=6):
    """ATR-adaptive zigzag. Returns (bottom_indices, top_indices)."""
    n = len(close)
    atr = compute_atr(high, low, close, atr_period)
    # Fill NaN ATR with first valid
    first_valid = np.where(~np.isnan(atr))[0]
    if len(first_valid) > 0:
        atr[:first_valid[0]] = atr[first_valid[0]]
    atr = np.nan_to_num(atr, nan=close.mean() * 0.01)

    bottoms, tops = [], []
    direction = 0  # 0=undecided, 1=looking for top, -1=looking for bottom
    last_high_idx = 0
    last_low_idx = 0
    last_high = high[0]
    last_low = low[0]
    last_tp_idx = -min_bars * 2

    for i in range(1, n):
        threshold = atr[i] * atr_mult

        if direction >= 0:  # looking for top or undecided
            if high[i] > last_high:
                last_high = high[i]
                last_high_idx = i
            if last_high - low[i] >= threshold and i - last_tp_idx >= min_bars:
                if last_high_idx != 0:
                    tops.append(last_high_idx)
                    last_tp_idx = last_high_idx
                direction = -1
                last_low = low[i]
                last_low_idx = i

        if direction <= 0:  # looking for bottom or undecided
            if low[i] < last_low:
                last_low = low[i]
                last_low_idx = i
            if high[i] - last_low >= threshold and i - last_tp_idx >= min_bars:
                if last_low_idx != 0:
                    bottoms.append(last_low_idx)
                    last_tp_idx = last_low_idx
                direction = 1
                last_high = high[i]
                last_high_idx = i

    return np.array(bottoms, dtype=int), np.array(tops, dtype=int)


def make_labels(n, tp_indices, proximity=3):
    """Binary labels: 1 if within `proximity` bars of a TP."""
    labels = np.zeros(n, dtype=np.float32)
    for idx in tp_indices:
        lo = max(0, idx - proximity)
        hi = min(n, idx + proximity + 1)
        labels[lo:hi] = 1.0
    return labels


# ═══════════════════════════════════════════════════════════════
#  ALIGNMENT: 15min → 1hour mapping
# ═══════════════════════════════════════════════════════════════

def build_hour_index_map(ts_15min, ts_1hour):
    """
    For each 15min timestamp, find the index of the latest 1hour bar
    at or before that time. Returns array of 1hour indices (or -1 if none).
    """
    ts_h = ts_1hour.values.astype(np.int64)
    ts_s = ts_15min.values.astype(np.int64)
    hour_idx = np.full(len(ts_s), -1, dtype=int)
    j = 0
    for i in range(len(ts_s)):
        while j < len(ts_h) - 1 and ts_h[j + 1] <= ts_s[i]:
            j += 1
        if ts_h[j] <= ts_s[i]:
            hour_idx[i] = j
    return hour_idx


# ═══════════════════════════════════════════════════════════════
#  DATASET CONSTRUCTION
# ═══════════════════════════════════════════════════════════════

def build_paired_windows(cfg):
    """
    Build aligned (short_window, long_window, label, meta) for all tickers.
    Returns dict with train/val/test splits.
    """
    print(f"\n  Building paired windows...")
    short_win = cfg["SHORT_WIN"]
    long_win = cfg["LONG_WIN"]
    prox = cfg["TP_PROXIMITY"]

    all_data = {"short": [], "long": [], "bottom_label": [], "top_label": [],
                "ticker": [], "bar_idx": [], "timestamp": []}

    for ticker in cfg["TICKERS"]:
        try:
            df_s = load_csv(ticker, cfg["SHORT_FREQ"], cfg)
            df_l = load_csv(ticker, cfg["LONG_FREQ"], cfg)
        except FileNotFoundError as e:
            print(f"    [SKIP] {ticker}: {e}")
            continue

        # Dynamics features on 1hour
        df_l = add_dynamics_features(df_l)

        # Feature columns
        s_cols = numeric_cols(df_s)
        l_cols = numeric_cols(df_l)

        s_feat = df_s[s_cols].values.astype(np.float32)
        l_feat = df_l[l_cols].values.astype(np.float32)

        # NaN → 0
        s_feat = np.nan_to_num(s_feat, nan=0.0, posinf=0.0, neginf=0.0)
        l_feat = np.nan_to_num(l_feat, nan=0.0, posinf=0.0, neginf=0.0)

        # Zigzag labels on 15min
        close_s = df_s["close"].values.astype(float)
        high_s = df_s["high"].values.astype(float) if "high" in df_s.columns else close_s
        low_s = df_s["low"].values.astype(float) if "low" in df_s.columns else close_s

        bottoms, tops = zigzag_atr(high_s, low_s, close_s,
                                   cfg["ZIGZAG_ATR_MULT"],
                                   cfg["ZIGZAG_ATR_PERIOD"],
                                   cfg["MIN_BARS_BETWEEN"])
        bottom_labels = make_labels(len(df_s), bottoms, prox)
        top_labels = make_labels(len(df_s), tops, prox)

        # Alignment map
        hour_idx_map = build_hour_index_map(df_s["timestamp"], df_l["timestamp"])

        # Build windows
        n_short = len(df_s)
        count = 0
        for i in range(short_win, n_short):
            hi = hour_idx_map[i]
            if hi < long_win:
                continue  # not enough 1hour history

            short_window = s_feat[i - short_win:i]  # (30, N_s)
            long_window = l_feat[hi - long_win + 1:hi + 1]  # (48, N_l)

            if short_window.shape[0] != short_win or long_window.shape[0] != long_win:
                continue

            all_data["short"].append(short_window)
            all_data["long"].append(long_window)
            all_data["bottom_label"].append(bottom_labels[i])
            all_data["top_label"].append(top_labels[i])
            all_data["ticker"].append(ticker)
            all_data["bar_idx"].append(i)
            all_data["timestamp"].append(str(df_s["timestamp"].iloc[i]))
            count += 1

        n_bot = int(bottom_labels[short_win:].sum())
        n_top = int(top_labels[short_win:].sum())
        print(f"    {ticker}: {count:,} samples, "
              f"bottoms={n_bot} tops={n_top}  "
              f"s_feat={s_feat.shape[1]} l_feat={l_feat.shape[1]}")

    # Convert to arrays
    X_short = np.array(all_data["short"], dtype=np.float32)
    X_long = np.array(all_data["long"], dtype=np.float32)
    y_bottom = np.array(all_data["bottom_label"], dtype=np.float32)
    y_top = np.array(all_data["top_label"], dtype=np.float32)
    tickers = np.array(all_data["ticker"])
    bar_indices = np.array(all_data["bar_idx"])
    timestamps = np.array(all_data["timestamp"])

    n = len(y_bottom)
    print(f"\n  Total: {n:,} samples")
    print(f"  Short shape: {X_short.shape}  Long shape: {X_long.shape}")
    print(f"  Bottom pos rate: {y_bottom.mean():.4f}  "
          f"Top pos rate: {y_top.mean():.4f}")

    # Time-based split (data already sorted by time within each ticker,
    # but we sort globally by bar_idx proxy — use index order since tickers processed sequentially)
    # Actually split per-ticker to preserve time ordering
    train_mask = np.zeros(n, dtype=bool)
    val_mask = np.zeros(n, dtype=bool)
    test_mask = np.zeros(n, dtype=bool)

    for ticker in cfg["TICKERS"]:
        t_mask = tickers == ticker
        t_idx = np.where(t_mask)[0]
        if len(t_idx) == 0:
            continue
        n_t = len(t_idx)
        tr_end = int(n_t * 0.70)
        va_end = int(n_t * 0.85)
        train_mask[t_idx[:tr_end]] = True
        val_mask[t_idx[tr_end:va_end]] = True
        test_mask[t_idx[va_end:]] = True

    print(f"  Train: {train_mask.sum():,}  Val: {val_mask.sum():,}  "
          f"Test: {test_mask.sum():,}")

    return {
        "X_short": X_short, "X_long": X_long,
        "y_bottom": y_bottom, "y_top": y_top,
        "tickers": tickers, "bar_indices": bar_indices,
        "timestamps": timestamps,
        "train_mask": train_mask, "val_mask": val_mask, "test_mask": test_mask,
        "n_short_feat": X_short.shape[2], "n_long_feat": X_long.shape[2],
    }


# ═══════════════════════════════════════════════════════════════
#  NORMALIZE + UNDERSAMPLE
# ═══════════════════════════════════════════════════════════════

def normalize_windows(X_train, *others):
    """Fit scaler on train, transform all. X shape: (N, T, F)."""
    n, t, f = X_train.shape
    flat = X_train.reshape(-1, f)
    mu = np.nanmean(flat, axis=0)
    sigma = np.nanstd(flat, axis=0) + 1e-8
    mu = np.nan_to_num(mu, nan=0.0)
    sigma = np.where(np.isnan(sigma) | (sigma < 1e-8), 1.0, sigma)

    results = []
    for arr in (X_train,) + others:
        normed = (arr - mu) / sigma
        np.nan_to_num(normed, copy=False, nan=0.0, posinf=0.0, neginf=0.0)
        results.append(normed)
    return tuple(results) + (mu, sigma)


def undersample_negatives(X_s, X_l, y, neg_ratio=3, seed=42):
    """Undersample negatives to neg_ratio:1."""
    pos_idx = np.where(y == 1)[0]
    neg_idx = np.where(y == 0)[0]
    n_pos = len(pos_idx)
    n_neg_keep = min(n_pos * neg_ratio, len(neg_idx))
    rng = np.random.RandomState(seed)
    neg_keep = rng.choice(neg_idx, size=n_neg_keep, replace=False)
    keep = np.sort(np.concatenate([pos_idx, neg_keep]))
    return X_s[keep], X_l[keep], y[keep]


# ═══════════════════════════════════════════════════════════════
#  MODEL
# ═══════════════════════════════════════════════════════════════

class MultiScaleCNN(nn.Module):
    def __init__(self, n_short_feat, n_long_feat):
        super().__init__()
        # Short branch (15min, 30 bars)
        self.short_branch = nn.Sequential(
            nn.Conv1d(n_short_feat, 32, kernel_size=5, padding=2),
            nn.ReLU(), nn.BatchNorm1d(32),
            nn.Conv1d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(), nn.BatchNorm1d(64),
            nn.Conv1d(64, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
        )
        # Long branch (1hour, 48 bars)
        self.long_branch = nn.Sequential(
            nn.Conv1d(n_long_feat, 32, kernel_size=7, padding=3),
            nn.ReLU(), nn.BatchNorm1d(32),
            nn.Conv1d(32, 64, kernel_size=5, padding=2),
            nn.ReLU(), nn.BatchNorm1d(64),
            nn.Conv1d(64, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
        )
        # Fusion
        self.head = nn.Sequential(
            nn.Linear(64, 32), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(32, 1),
        )

    def forward(self, x_short, x_long):
        # Input: (B, T, F) → Conv1d wants (B, F, T)
        s = self.short_branch(x_short.transpose(1, 2))
        l = self.long_branch(x_long.transpose(1, 2))
        fused = torch.cat([s, l], dim=1)
        return self.head(fused).squeeze(-1)


class PairedDataset(Dataset):
    def __init__(self, X_short, X_long, y):
        self.xs = torch.tensor(X_short, dtype=torch.float32)
        self.xl = torch.tensor(X_long, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        return self.xs[i], self.xl[i], self.y[i]


# ═══════════════════════════════════════════════════════════════
#  TRAINING
# ═══════════════════════════════════════════════════════════════

def train_model(X_s_tr, X_l_tr, y_tr, X_s_va, X_l_va, y_va,
                n_short_feat, n_long_feat, cfg, task):
    """Train MultiScaleCNN for one task."""
    print(f"\n    Training MultiScaleCNN ({task})...")
    # Undersample training
    X_s_tr_u, X_l_tr_u, y_tr_u = undersample_negatives(
        X_s_tr, X_l_tr, y_tr, cfg["NEG_RATIO"])
    print(f"    After undersample: {len(y_tr_u):,} "
          f"(pos={y_tr_u.sum():.0f}, neg={len(y_tr_u)-y_tr_u.sum():.0f})")

    model = MultiScaleCNN(n_short_feat, n_long_feat).to(DEVICE)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"    Params: {n_params:,}")

    pos_weight = torch.tensor(
        [(1 - y_tr_u.mean()) / (y_tr_u.mean() + 1e-8)],
        dtype=torch.float32).to(DEVICE)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    optimizer = torch.optim.Adam(model.parameters(), lr=cfg["LR"],
                                  weight_decay=cfg["WD"])
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=cfg["SCHED_PATIENCE"])

    train_dl = DataLoader(PairedDataset(X_s_tr_u, X_l_tr_u, y_tr_u),
                          batch_size=cfg["BATCH"], shuffle=True,
                          num_workers=0, pin_memory=True)
    val_dl = DataLoader(PairedDataset(X_s_va, X_l_va, y_va),
                        batch_size=cfg["BATCH"] * 4, shuffle=False,
                        num_workers=0, pin_memory=True)

    best_loss = float("inf")
    best_state = None
    patience_ctr = 0
    best_epoch = 0

    for epoch in range(cfg["EPOCHS"]):
        model.train()
        t_loss = 0.0
        for xs, xl, yb in train_dl:
            xs, xl, yb = xs.to(DEVICE), xl.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            logits = model(xs, xl)
            if torch.isnan(logits).any():
                continue
            loss = criterion(logits, yb)
            if torch.isnan(loss):
                continue
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), cfg["GRAD_CLIP"])
            optimizer.step()
            t_loss += loss.item() * len(yb)
        t_loss /= max(len(y_tr_u), 1)

        model.eval()
        v_loss = 0.0
        with torch.no_grad():
            for xs, xl, yb in val_dl:
                xs, xl, yb = xs.to(DEVICE), xl.to(DEVICE), yb.to(DEVICE)
                logits = model(xs, xl)
                v_loss += criterion(logits, yb).item() * len(yb)
        v_loss /= max(len(y_va), 1)

        scheduler.step(v_loss)
        is_best = v_loss < best_loss
        if is_best:
            best_loss = v_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_ctr = 0
            best_epoch = epoch + 1
        else:
            patience_ctr += 1

        lr = optimizer.param_groups[0]["lr"]
        if (epoch + 1) % 10 == 0 or is_best or patience_ctr == cfg["PATIENCE"]:
            tag = "  *best" if is_best else ""
            print(f"      E{epoch+1:3d} t={t_loss:.6f} v={v_loss:.6f} "
                  f"lr={lr:.1e} p={patience_ctr}{tag}")
        if patience_ctr >= cfg["PATIENCE"]:
            print(f"      Early stop (best={best_epoch})")
            break

    if best_state:
        model.load_state_dict(best_state)
    model.to(DEVICE)
    return model


def predict(model, X_s, X_l, batch_size=1024):
    model.eval()
    dl = DataLoader(PairedDataset(X_s, X_l, np.zeros(len(X_s))),
                    batch_size=batch_size, shuffle=False,
                    num_workers=0, pin_memory=True)
    preds = []
    with torch.no_grad():
        for xs, xl, _ in dl:
            logits = model(xs.to(DEVICE), xl.to(DEVICE)).cpu().numpy()
            probs = 1.0 / (1.0 + np.exp(-np.clip(logits, -20, 20)))
            preds.append(np.nan_to_num(probs, nan=0.5))
    return np.concatenate(preds)


# ═══════════════════════════════════════════════════════════════
#  LEVEL 1: DETECTION METRICS
# ═══════════════════════════════════════════════════════════════

def eval_detection(probs, y_true, thresholds, baseline_rate):
    """Precision, recall, F1 at each threshold + AUC."""
    auc = roc_auc_score(y_true, probs) if y_true.sum() > 0 and (1 - y_true).sum() > 0 else 0.5
    results = {"auc": auc, "n": len(y_true),
               "n_pos": int(y_true.sum()),
               "baseline_rate": baseline_rate}
    for th in thresholds:
        pred = (probs >= th).astype(int)
        tp = int(((pred == 1) & (y_true == 1)).sum())
        fp = int(((pred == 1) & (y_true == 0)).sum())
        fn = int(((pred == 0) & (y_true == 1)).sum())
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0
        rec = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0
        n_pred = tp + fp
        # z-score: precision vs baseline rate
        se = np.sqrt(baseline_rate * (1 - baseline_rate) / n_pred) if n_pred > 0 and 0 < baseline_rate < 1 else 1
        z = (prec - baseline_rate) / se if se > 0 and n_pred > 0 else 0
        results[th] = {"precision": prec, "recall": rec, "f1": f1,
                       "n_pred": n_pred, "tp": tp, "z": z}
    return results


def print_detection(results, task, single_auc):
    print(f"\n  {task.upper()} — Detection Metrics  "
          f"(AUC={results['auc']:.3f}  single-scale={single_auc:.3f}  "
          f"Δ={results['auc']-single_auc:+.3f})")
    print(f"  {'Th':>5} {'Prec':>8} {'Recall':>8} {'F1':>8} "
          f"{'N_pred':>8} {'TP':>6} {'z':>7}")
    print(f"  {'-'*52}")
    for th in sorted(k for k in results if isinstance(k, float)):
        r = results[th]
        sig = "*" if abs(r["z"]) > 1.96 else ""
        print(f"  {th:>5.1f} {r['precision']*100:>7.2f}% "
              f"{r['recall']*100:>7.1f}% {r['f1']*100:>7.2f}% "
              f"{r['n_pred']:>8} {r['tp']:>6} {r['z']:>+6.2f}{sig}")


# ═══════════════════════════════════════════════════════════════
#  LEVEL 2: MFE/MAE ANALYSIS
# ═══════════════════════════════════════════════════════════════

def compute_mfe_mae_for_candidates(cfg, probs, tickers, bar_indices,
                                    test_mask, task, threshold=0.5):
    """Compute MFE/MAE for multi-scale CNN candidates."""
    direction = "long" if task == "bottom" else "short"
    cand_mask = test_mask & (probs >= threshold)
    cand_idx = np.where(cand_mask)[0]

    if len(cand_idx) == 0:
        return {}

    # Group by ticker
    results = {}
    for mb in cfg["MFE_MAX_BARS"]:
        all_mfe, all_mae, all_probs = [], [], []
        for ticker in cfg["TICKERS"]:
            try:
                df = load_csv(ticker, cfg["SHORT_FREQ"], cfg)
            except FileNotFoundError:
                continue
            close = df["close"].values.astype(float)
            high = df["high"].values.astype(float) if "high" in df.columns else close
            low = df["low"].values.astype(float) if "low" in df.columns else close

            t_cand = cand_idx[(tickers[cand_idx] == ticker)]
            for ci in t_cand:
                bi = bar_indices[ci]
                if bi + 1 >= len(close):
                    continue
                end = min(bi + mb + 1, len(close))
                if bi + 1 >= end:
                    continue
                seg_h = high[bi + 1:end]
                seg_l = low[bi + 1:end]
                entry = close[bi]
                if direction == "long":
                    mfe = max((np.max(seg_h) - entry) / entry, 0)
                    mae = max((entry - np.min(seg_l)) / entry, 0)
                else:
                    mfe = max((entry - np.min(seg_l)) / entry, 0)
                    mae = max((np.max(seg_h) - entry) / entry, 0)
                all_mfe.append(mfe)
                all_mae.append(mae)
                all_probs.append(probs[ci])

        all_mfe = np.array(all_mfe)
        all_mae = np.array(all_mae)
        all_probs = np.array(all_probs)
        mfe_med = np.median(all_mfe) if len(all_mfe) > 0 else 0
        mae_med = np.median(all_mae) if len(all_mae) > 0 else 0
        ratio = mfe_med / mae_med if mae_med > 0 else 999

        results[mb] = {
            "n": len(all_mfe), "mfe_med": float(mfe_med),
            "mae_med": float(mae_med), "ratio": float(ratio),
            "mfe": all_mfe, "mae": all_mae, "probs": all_probs,
        }

        print(f"    MB={mb}: N={len(all_mfe):,}  "
              f"MFE_med={mfe_med*100:.3f}%  MAE_med={mae_med*100:.3f}%  "
              f"ratio={ratio:.3f}")

    # Probability bucketed analysis (using MB=24)
    best_mb = 24 if 24 in results else list(results.keys())[0]
    d = results[best_mb]
    if d["n"] > 0:
        print(f"\n    Prob-bucketed MFE/MAE (MB={best_mb}):")
        print(f"    {'Bucket':>12} {'N':>6} {'MFE_med':>9} {'MAE_med':>9} "
              f"{'Ratio':>7}")
        print(f"    {'-'*48}")
        buckets = [(0.5, 0.6), (0.6, 0.7), (0.7, 0.8), (0.8, 1.01)]
        for lo, hi in buckets:
            mask = (d["probs"] >= lo) & (d["probs"] < hi)
            n_b = mask.sum()
            if n_b < 5:
                print(f"    [{lo:.1f}-{hi:.1f}) {n_b:>6}  —")
                continue
            mfe_b = np.median(d["mfe"][mask])
            mae_b = np.median(d["mae"][mask])
            r_b = mfe_b / mae_b if mae_b > 0 else 999
            print(f"    [{lo:.1f}-{hi:.1f}) {n_b:>6} "
                  f"{mfe_b*100:>8.3f}% {mae_b*100:>8.3f}% {r_b:>7.3f}")

    return results


# ═══════════════════════════════════════════════════════════════
#  LEVEL 3: META-LABEL (LightGBM)
# ═══════════════════════════════════════════════════════════════

def triple_barrier_vectorized(close_arr, high_arr, low_arr,
                              indices, direction, tp_pct, sl_pct, max_bars):
    """
    Vectorized triple-barrier labelling — processes ALL candidates simultaneously.

    For each horizon step h=1..max_bars, computes TP/SL hits across all
    still-open positions in a single numpy pass.  Turns O(N * max_bars) Python
    iterations into O(max_bars) numpy vectorised operations.

    Returns: labels (int array), pnls (float array)
    """
    n_price = len(close_arr)
    n_cand = len(indices)

    entries = close_arr[indices]                       # (n_cand,)
    labels = np.zeros(n_cand, dtype=np.int32)
    pnls = np.zeros(n_cand, dtype=np.float64)
    still_open = np.ones(n_cand, dtype=bool)

    for h in range(1, max_bars + 1):
        if not still_open.any():
            break

        future_idx = indices + h
        valid = still_open & (future_idx < n_price)
        if not valid.any():
            continue

        vi = np.where(valid)[0]                        # indices into candidates
        fi = future_idx[vi]                            # indices into price arrays
        ent = entries[vi]

        if direction == "long":
            tp_hit = (high_arr[fi] - ent) / ent >= tp_pct
            sl_hit = (ent - low_arr[fi]) / ent >= sl_pct
        else:
            tp_hit = (ent - low_arr[fi]) / ent >= tp_pct
            sl_hit = (high_arr[fi] - ent) / ent >= sl_pct

        # Both hit same bar → conservative: SL wins
        both = tp_hit & sl_hit
        pure_tp = tp_hit & ~sl_hit
        pure_sl = sl_hit & ~tp_hit

        # TP exits
        tp_idx = vi[pure_tp]
        labels[tp_idx] = 1
        pnls[tp_idx] = tp_pct
        still_open[tp_idx] = False

        # SL exits (including both-hit)
        sl_idx = vi[pure_sl | both]
        labels[sl_idx] = 0
        pnls[sl_idx] = -sl_pct
        still_open[sl_idx] = False

    # Timeout exits — mark-to-market PnL
    to_mask = still_open
    if to_mask.any():
        to_idx = np.where(to_mask)[0]
        exit_idx = np.minimum(indices[to_idx] + max_bars, n_price - 1)
        exit_p = close_arr[exit_idx]
        if direction == "long":
            pnls[to_idx] = (exit_p - entries[to_idx]) / entries[to_idx]
        else:
            pnls[to_idx] = (entries[to_idx] - exit_p) / entries[to_idx]
        # labels already 0

    return labels, pnls


def build_meta_features(X_short, X_long, probs):
    """Flatten last few bars + CNN prob as LGB features."""
    n = len(probs)
    # Short: last 5 bars flattened
    s_last = X_short[:, -5:, :].reshape(n, -1)
    # Long: last 3 bars flattened
    l_last = X_long[:, -3:, :].reshape(n, -1)
    # CNN prob
    prob_col = probs.reshape(-1, 1)
    return np.nan_to_num(np.hstack([s_last, l_last, prob_col]),
                         nan=0.0, posinf=0.0, neginf=0.0)


def run_meta_label(cfg, probs, X_short, X_long, tickers, bar_indices,
                   test_mask, task, threshold):
    """Level 3: LightGBM meta-label on multi-scale CNN candidates.

    Fixed: pre-loads ticker CSVs once (8 reads, not 50k), then uses
    vectorized triple-barrier labelling per ticker.
    """
    if not HAS_LGB:
        print("    [SKIP] LightGBM not installed")
        return {}

    direction = "long" if task == "bottom" else "short"
    cand_mask = test_mask & (probs >= threshold)
    cand_idx = np.where(cand_mask)[0]
    if len(cand_idx) < 50:
        print(f"    [SKIP] Only {len(cand_idx)} candidates")
        return {}

    print(f"    Candidates: {len(cand_idx):,}")

    # ── Step A: Pre-load all ticker price data ONCE ──
    t0 = time.time()
    ticker_data = {}  # ticker → (close, high, low)
    for ticker in cfg["TICKERS"]:
        try:
            df = load_csv(ticker, cfg["SHORT_FREQ"], cfg)
            close = df["close"].values.astype(np.float64)
            high = df["high"].values.astype(np.float64) if "high" in df.columns else close.copy()
            low = df["low"].values.astype(np.float64) if "low" in df.columns else close.copy()
            ticker_data[ticker] = (close, high, low)
        except FileNotFoundError:
            pass
    print(f"    Loaded {len(ticker_data)} ticker CSVs  ({time.time()-t0:.2f}s)")

    # ── Step B: Vectorized triple-barrier labelling per ticker ──
    t1 = time.time()
    all_labels = np.full(len(cand_idx), -1, dtype=np.int32)
    all_pnls = np.zeros(len(cand_idx), dtype=np.float64)
    valid_mask = np.zeros(len(cand_idx), dtype=bool)

    cand_tickers = tickers[cand_idx]
    cand_bars = bar_indices[cand_idx]

    for ticker, (close, high, low) in ticker_data.items():
        t_mask = cand_tickers == ticker
        if not t_mask.any():
            continue
        t_positions = np.where(t_mask)[0]       # positions within cand_idx
        t_bar_idx = cand_bars[t_positions]       # bar indices into price arrays

        # Filter out-of-range
        ok = (t_bar_idx >= 1) & (t_bar_idx < len(close) - 1)
        if not ok.any():
            continue
        t_positions = t_positions[ok]
        t_bar_idx = t_bar_idx[ok]

        labs, pnls_v = triple_barrier_vectorized(
            close, high, low, t_bar_idx, direction,
            cfg["META_TP"], cfg["META_SL"], cfg["META_MB"])

        all_labels[t_positions] = labs
        all_pnls[t_positions] = pnls_v
        valid_mask[t_positions] = True

    # Keep only valid candidates
    valid_cand = cand_idx[valid_mask]
    labels = all_labels[valid_mask].astype(np.float32)
    pnls = all_pnls[valid_mask].astype(np.float32)
    print(f"    Triple-barrier labelling: {len(labels):,} trades  ({time.time()-t1:.2f}s)")

    if len(valid_cand) < 50:
        print(f"    [SKIP] Only {len(valid_cand)} valid candidates")
        return {}

    # ── Step C: Build features ──
    t2 = time.time()
    X_meta = build_meta_features(X_short[valid_cand],
                                 X_long[valid_cand],
                                 probs[valid_cand])
    print(f"    Feature matrix: {X_meta.shape}  ({time.time()-t2:.2f}s)")

    # ── Step D: Train/test split + LightGBM ──
    t3 = time.time()
    n_c = len(labels)
    split = int(n_c * 0.7)
    X_tr, y_tr, pnl_tr = X_meta[:split], labels[:split], pnls[:split]
    X_te, y_te, pnl_te = X_meta[split:], labels[split:], pnls[split:]

    base_rate = float(y_te.mean())
    print(f"    Meta-label: train={len(y_tr):,} test={len(y_te):,} "
          f"base_wr={base_rate*100:.1f}%")

    val_split = int(len(y_tr) * 0.8)
    params = dict(cfg["LGB_PARAMS"])
    n_est = params.pop("n_estimators", 500)
    model = lgb.LGBMClassifier(n_estimators=n_est, **params)
    model.fit(X_tr[:val_split], y_tr[:val_split],
              eval_set=[(X_tr[val_split:], y_tr[val_split:])],
              callbacks=[lgb.early_stopping(30, verbose=False),
                         lgb.log_evaluation(0)])
    print(f"    LightGBM training  ({time.time()-t3:.2f}s)")

    # ── Step E: Evaluate ──
    t4 = time.time()
    lgb_probs = model.predict_proba(X_te)[:, 1]

    results = {"base_rate": base_rate, "n_test": len(y_te)}
    print(f"\n    {'Th':>5} {'WR':>8} {'N':>6} {'PF':>7} {'TotPnL':>9}")
    print(f"    {'-'*42}")
    for th in cfg["LGB_THRESHOLDS"]:
        m = lgb_probs >= th
        n_t = int(m.sum())
        if n_t == 0:
            results[th] = {"wr": 0, "n": 0, "pf": 0, "pnl": 0}
            continue
        wr = float(y_te[m].mean())
        ps = pnl_te[m]
        gp = ps[ps > 0].sum()
        gl = abs(ps[ps < 0].sum())
        pf = gp / gl if gl > 0 else (999 if gp > 0 else 0)
        results[th] = {"wr": wr, "n": n_t,
                       "pf": float(pf), "pnl": float(ps.sum())}
        print(f"    {th:>5.2f} {wr*100:>7.1f}% {n_t:>6} "
              f"{pf:>7.2f} {ps.sum()*100:>8.3f}%")
    print(f"    Evaluation  ({time.time()-t4:.2f}s)")

    import joblib
    meta_dir = os.path.join(cfg["OUTPUT_DIR"],
                            f"models/layer3/trade_filter/lgb_{task}_v1")
    os.makedirs(meta_dir, exist_ok=True)
    joblib.dump(model, os.path.join(meta_dir, "weights.joblib"))
    with open(os.path.join(meta_dir, "meta.json"), "w") as _mf:
        json.dump({
            "adapter": "lightgbm",
            "output": "probability",
            "task": task,
            "direction": "long" if task == "bottom" else "short",
            "metrics": {
                "base_wr": float(base_rate),
                "wr_at_0.45": float(results.get(0.45, {}).get("wr", 0)),
                "wr_at_0.50": float(results.get(0.50, {}).get("wr", 0)),
                "n_test": len(y_te),
            }
        }, _mf, indent=2)
    print(f"    Saved Meta LGB ({task}): {meta_dir}/weights.joblib")

    return results


# ═══════════════════════════════════════════════════════════════
#  SAVE PREDICTIONS
# ═══════════════════════════════════════════════════════════════

def save_predictions(probs, tickers, bar_indices, timestamps,
                     test_mask, cfg, task):
    rows = []
    test_idx = np.where(test_mask)[0]
    for i in test_idx:
        rows.append({
            "ticker": tickers[i],
            "bar_index": int(bar_indices[i]),
            "timestamp": timestamps[i],
            "prob": float(probs[i]),
        })
    df = pd.DataFrame(rows)
    path = os.path.join(cfg["OUTPUT_DIR"],
                        f"multiscale_cnn_{task}_predictions.csv")
    df.to_csv(path, index=False)
    print(f"  Saved: {path} ({len(df)} rows)")


# ═══════════════════════════════════════════════════════════════
#  MAIN
# ═══════════════════════════════════════════════════════════════

def main():
    cfg = CFG
    print("=" * 78)
    print("  Multi-Scale CNN for Turning Point Detection")
    print(f"  Short: {cfg['SHORT_WIN']}×{cfg['SHORT_FREQ']}  "
          f"Long: {cfg['LONG_WIN']}×{cfg['LONG_FREQ']}")
    print(f"  Device: {DEVICE}")
    print("=" * 78)

    os.makedirs(cfg["OUTPUT_DIR"], exist_ok=True)

    # ── Build data ──
    data = build_paired_windows(cfg)

    X_s = data["X_short"]
    X_l = data["X_long"]
    tr = data["train_mask"]
    va = data["val_mask"]
    te = data["test_mask"]

    # Normalize short and long branches separately
    X_s_tr, X_s_va, X_s_te, _, _ = normalize_windows(
        X_s[tr], X_s[va], X_s[te])
    X_l_tr, X_l_va, X_l_te, _, _ = normalize_windows(
        X_l[tr], X_l[va], X_l[te])

    # Reassemble full arrays (needed for prediction indexing)
    X_s_full = np.zeros_like(X_s)
    X_l_full = np.zeros_like(X_l)
    X_s_full[tr] = X_s_tr
    X_s_full[va] = X_s_va
    X_s_full[te] = X_s_te
    X_l_full[tr] = X_l_tr
    X_l_full[va] = X_l_va
    X_l_full[te] = X_l_te

    all_results = {}

    for task in ["bottom", "top"]:
        y_col = data["y_bottom"] if task == "bottom" else data["y_top"]
        print(f"\n{'#'*78}")
        print(f"  TASK: {task.upper()}")
        print(f"{'#'*78}")

        y_tr = y_col[tr]
        y_va = y_col[va]
        y_te = y_col[te]

        det = {}
        mfe_results = {}
        train_time = 0.0

        skip_cfg = cfg.get("SKIP_L1_L2", False)
        skip_this = (skip_cfg is True or
                     (isinstance(skip_cfg, list) and task in skip_cfg))

        if skip_this:
            # ── Load saved predictions instead of training ──
            pred_path = os.path.join(cfg["OUTPUT_DIR"],
                                     f"multiscale_cnn_{task}_predictions.csv")
            ckpt_path = os.path.join(cfg["OUTPUT_DIR"],
                                     f"multiscale_cnn_{task}.pt")
            if not os.path.exists(pred_path):
                print(f"  [ERROR] {pred_path} not found, cannot skip L1/L2")
                continue

            print(f"  Loading saved predictions: {pred_path}")
            pred_df = pd.read_csv(pred_path)
            probs = np.zeros(len(y_col))
            # Map predictions back to test indices
            test_idx_arr = np.where(te)[0]
            test_tickers = data["tickers"][test_idx_arr]
            test_bars = data["bar_indices"][test_idx_arr]
            for _, row in pred_df.iterrows():
                matches = ((test_tickers == row["ticker"]) &
                           (test_bars == int(row["bar_index"])))
                mi = np.where(matches)[0]
                if len(mi) > 0:
                    probs[test_idx_arr[mi[0]]] = row["prob"]
            print(f"  Loaded {len(pred_df):,} predictions, "
                  f"mapped to {(probs[te] > 0).sum():,} test samples")
            print(f"\n  ═══ SKIPPING LEVELS 1-2 (already completed) ═══")

        else:
            # ── Train ──
            t0 = time.time()
            model = train_model(X_s_tr, X_l_tr, y_tr,
                                X_s_va, X_l_va, y_va,
                                data["n_short_feat"], data["n_long_feat"],
                                cfg, task)
            train_time = time.time() - t0
            print(f"    Training time: {train_time:.1f}s")

            # Save model
            ckpt_path = os.path.join(cfg["OUTPUT_DIR"],
                                     f"multiscale_cnn_{task}.pt")
            torch.save(model.state_dict(), ckpt_path)
            print(f"    Saved: {ckpt_path}")

            # ── Predict on full dataset ──
            probs = np.zeros(len(y_col))
            probs[te] = predict(model, X_s_te, X_l_te)
            probs[va] = predict(model, X_s_va, X_l_va)

            # ── Save predictions ──
            save_predictions(probs, data["tickers"], data["bar_indices"],
                             data["timestamps"], te, cfg, task)

            # ════════════════════════════════════════════
            # LEVEL 1: Detection Metrics
            # ════════════════════════════════════════════
            print(f"\n  ═══ LEVEL 1: DETECTION METRICS ═══")

            baseline_rate = y_te.mean()
            det = eval_detection(probs[te], y_te, cfg["DET_THRESHOLDS"],
                                 baseline_rate)
            single_auc = cfg["SINGLE_BASELINES"][f"{task}_auc"]
            print_detection(det, task, single_auc)

            # ════════════════════════════════════════════
            # LEVEL 2: MFE/MAE Analysis
            # ════════════════════════════════════════════
            print(f"\n  ═══ LEVEL 2: MFE/MAE ANALYSIS ═══")

            mfe_results = compute_mfe_mae_for_candidates(
                cfg, probs, data["tickers"], data["bar_indices"],
                te, task, threshold=0.5)

        # ════════════════════════════════════════════
        # LEVEL 3: Meta-Label
        # ════════════════════════════════════════════
        print(f"\n  ═══ LEVEL 3: META-LABEL INTEGRATION ═══")

        # Pick threshold with best MFE/MAE ratio and N>500
        best_th = 0.5
        if mfe_results:
            mb24 = mfe_results.get(24, {})
            if mb24 and mb24.get("n", 0) > 0:
                # Try higher thresholds
                for test_th in [0.5, 0.6, 0.7]:
                    cand_n = (te & (probs >= test_th)).sum()
                    if cand_n > 500:
                        best_th = test_th

        print(f"  Using CNN threshold={best_th} for meta-label")
        t_l3 = time.time()
        meta = run_meta_label(cfg, probs, X_s_full, X_l_full,
                              data["tickers"], data["bar_indices"],
                              te, task, best_th)
        print(f"  Level 3 total: {time.time()-t_l3:.2f}s")

        all_results[task] = {
            "detection": {
                "auc": det.get("auc", 0),
                "n": det.get("n", 0),
                "thresholds": {
                    str(k): v for k, v in det.items()
                    if isinstance(k, float)
                },
            } if det else {},
            "mfe_mae": {
                str(k): {kk: float(vv) if not isinstance(vv, np.ndarray) else None
                         for kk, vv in v.items()}
                for k, v in mfe_results.items()
            } if mfe_results else {},
            "meta_label": {
                str(k): (v if not isinstance(v, dict)
                         else {kk: float(vv) if isinstance(vv, (float, np.floating))
                               else int(vv) if isinstance(vv, (int, np.integer))
                               else vv
                               for kk, vv in v.items()})
                for k, v in meta.items()
            } if meta else {},
            "train_time": train_time,
        }

    # ═══════════════════════════════════════════════════════════
    #  MERGE WITH PREVIOUSLY SAVED RESULTS (for skipped levels)
    # ═══════════════════════════════════════════════════════════
    prev_json = os.path.join(cfg["OUTPUT_DIR"], "multiscale_cnn_results.json")
    if os.path.exists(prev_json):
        try:
            with open(prev_json) as f:
                prev = json.load(f)
            for task in ["bottom", "top"]:
                if task in prev and task in all_results:
                    cur = all_results[task]
                    old = prev[task]
                    # Fill in missing sections from previous run
                    if not cur.get("detection") and old.get("detection"):
                        cur["detection"] = old["detection"]
                        print(f"  [INFO] Loaded {task} L1 detection from previous run")
                    if not cur.get("mfe_mae") and old.get("mfe_mae"):
                        cur["mfe_mae"] = old["mfe_mae"]
                        print(f"  [INFO] Loaded {task} L2 MFE/MAE from previous run")
        except Exception:
            pass

    # ═══════════════════════════════════════════════════════════
    #  META-LABEL SUMMARY (key results from Level 3)
    # ═══════════════════════════════════════════════════════════
    print(f"\n{'='*78}")
    print(f"  ★ LEVEL 3 META-LABEL SUMMARY")
    print(f"{'='*78}")
    for task in ["bottom", "top"]:
        r = all_results.get(task, {})
        meta = r.get("meta_label", {})
        if not meta:
            print(f"\n  {task.upper()}: not available")
            continue
        base = meta.get("base_rate", 0)
        n_test = meta.get("n_test", 0)
        direction = "LONG" if task == "bottom" else "SHORT"
        print(f"\n  {task.upper()} ({direction}):  base_wr={base*100:.1f}%  n_test={n_test}")
        print(f"  {'Th':>6} {'WR':>8} {'N':>7} {'PF':>7} {'TotPnL':>10} {'Δ vs base':>10}")
        print(f"  {'-'*52}")
        for th_key in sorted(k for k in meta if k not in ("base_rate", "n_test")):
            th_v = meta[th_key]
            if not isinstance(th_v, dict):
                continue
            wr = th_v.get("wr", 0)
            n_t = th_v.get("n", 0)
            pf = th_v.get("pf", 0)
            pnl = th_v.get("pnl", 0)
            delta = wr - base if n_t > 0 else 0
            mark = "✓" if wr > 0.55 and n_t >= 100 else ""
            print(f"  {th_key:>6} {wr*100:>7.1f}% {n_t:>7} {pf:>7.2f} "
                  f"{pnl*100:>9.1f}% {delta*100:>+9.1f}pp {mark}")

    # ═══════════════════════════════════════════════════════════
    #  FINAL COMPARISON TABLE
    # ═══════════════════════════════════════════════════════════
    print(f"\n{'='*78}")
    print(f"  ★ SINGLE-SCALE vs MULTI-SCALE CNN COMPARISON")
    print(f"{'='*78}")

    sb = cfg["SINGLE_BASELINES"]
    print(f"\n  ┌{'─'*68}┐")
    print(f"  │ {'Metric':<24} │ {'Single-Scale':>13} │ "
          f"{'Multi-Scale':>12} │ {'Δ':>10} │")
    print(f"  ├{'─'*68}┤")

    rows = []
    for task in ["bottom", "top"]:
        r = all_results.get(task, {})
        det = r.get("detection", {})
        mfe = r.get("mfe_mae", {})
        meta = r.get("meta_label", {})

        # AUC
        ms_auc = det.get("auc", 0)
        ss_auc = sb[f"{task}_auc"]
        rows.append((f"{task.title()} AUC", ss_auc, ms_auc))

        # MFE/MAE @24
        ms_ratio = 0
        if "24" in mfe and mfe["24"].get("ratio") is not None:
            ms_ratio = mfe["24"]["ratio"]
        ss_ratio = sb[f"{task}_mfe_mae_24"]
        rows.append((f"{task.title()} MFE/MAE @24", ss_ratio, ms_ratio))

        # LGB WR @0.5
        ms_wr = 0
        if meta:
            th_data = meta.get("0.5") or meta.get(0.5)
            if th_data and isinstance(th_data, dict):
                ms_wr = th_data.get("wr", 0)
        ss_wr = sb[f"{task}_lgb_wr_05"]
        rows.append((f"{task.title()} LGB WR @0.5", ss_wr, ms_wr))

    for name, ss, ms in rows:
        ss_s = f"{ss:.3f}" if ss < 1 else f"{ss*100:.1f}%"
        if ms == 0 and ss != 0:
            # Metric wasn't computed this run and not loaded from previous
            print(f"  │ {name:<24} │ {ss_s:>13} │ {'—':>12} │ {'—':>10} │")
        else:
            delta = ms - ss
            ms_s = f"{ms:.3f}" if ms < 1 else f"{ms*100:.1f}%"
            d_s = f"{delta:+.3f}" if abs(delta) < 1 else f"{delta*100:+.1f}pp"
            print(f"  │ {name:<24} │ {ss_s:>13} │ {ms_s:>12} │ {d_s:>10} │")

    print(f"  └{'─'*68}┘")

    # Conclusions
    print(f"\n  CONCLUSIONS:")
    for task in ["bottom", "top"]:
        r = all_results.get(task, {})
        det = r.get("detection", {})
        ms_auc = det.get("auc", 0)
        ss_auc = sb[f"{task}_auc"]

        if ms_auc == 0:
            print(f"    — {task.upper()}: AUC not available (L1 skipped or not run)")
        elif ms_auc > ss_auc + 0.01:
            print(f"    ✓ {task.upper()}: Multi-scale improves AUC "
                  f"({ss_auc:.3f} → {ms_auc:.3f})")
        elif ms_auc > ss_auc - 0.01:
            print(f"    ~ {task.upper()}: Multi-scale AUC comparable "
                  f"({ss_auc:.3f} → {ms_auc:.3f})")
        else:
            print(f"    ✗ {task.upper()}: Multi-scale AUC declined "
                  f"({ss_auc:.3f} → {ms_auc:.3f})")

        mfe = r.get("mfe_mae", {})
        if "24" in mfe and mfe["24"].get("ratio"):
            ratio = mfe["24"]["ratio"]
            ss_r = sb[f"{task}_mfe_mae_24"]
            if ratio > ss_r:
                print(f"    ✓ {task.upper()}: MFE/MAE improved "
                      f"({ss_r:.3f} → {ratio:.3f})")

    # Save
    json_path = os.path.join(cfg["OUTPUT_DIR"],
                             "multiscale_cnn_results.json")
    # Clean for JSON
    save = {}
    for task, r in all_results.items():
        save[task] = {
            "detection": r.get("detection", {}),
            "mfe_mae": r.get("mfe_mae", {}),
            "meta_label": r.get("meta_label", {}),
            "train_time": r.get("train_time", 0),
        }
    with open(json_path, "w") as f:
        json.dump(save, f, indent=2, default=str)
    print(f"\n  Saved: {json_path}")
    print("\n  Done!")


if __name__ == "__main__":
    main()

In [ ]:
import torch, os, numpy as np

task = "bottom"
cfg = CFG

print("Building paired windows...")
data = build_paired_windows(cfg)

# Step 2: Normalize
print("Normalizing...")
tr = data["train_mask"]
va = data["val_mask"]
te = data["test_mask"]

X_s_tr, X_s_va, X_s_te, s_mu, s_sigma = normalize_windows(
    data["X_short"][tr], data["X_short"][va], data["X_short"][te])
X_l_tr, X_l_va, X_l_te, l_mu, l_sigma = normalize_windows(
    data["X_long"][tr], data["X_long"][va], data["X_long"][te])

print("Training bottom CNN...")
model = train_model(X_s_tr, X_l_tr, data["y_bottom"][tr],
                    X_s_va, X_l_va, data["y_bottom"][va],
                    data["n_short_feat"], data["n_long_feat"],
                    cfg, task)

ckpt_path = os.path.join(cfg["OUTPUT_DIR"], "multiscale_cnn_bottom.pt")
checkpoint = {
    "model_state_dict": model.state_dict(),
    "n_short_feat": data["n_short_feat"],
    "n_long_feat": data["n_long_feat"],
    "short_win": cfg["SHORT_WIN"],
    "long_win": cfg["LONG_WIN"],
    "s_mu": torch.tensor(s_mu, dtype=torch.float32),
    "s_sigma": torch.tensor(s_sigma, dtype=torch.float32),
    "l_mu": torch.tensor(l_mu, dtype=torch.float32),
    "l_sigma": torch.tensor(l_sigma, dtype=torch.float32),
    "task": task,
}
torch.save(checkpoint, ckpt_path)

ck = torch.load(ckpt_path, map_location="cpu", weights_only=True)
nan_found = any(torch.isnan(v).any()
                for v in ck["model_state_dict"].values()
                if v.is_floating_point())
print(f"{'❌ NaN found' if nan_found else '✅ Clean'}: {ckpt_path}")
print(f"keys: {list(ck.keys())}")
print(f"n_short_feat={ck['n_short_feat']}  n_long_feat={ck['n_long_feat']}")